# build-af3-from-scratch — Overview · 项目总览

A walk-through of how the chapters compose into the full AlphaFold 3 model.
This notebook loads the bundled 7r6r protein, runs inference end-to-end on
CPU / MPS, and prints the pLDDT / pTM scores.

本 notebook 演示如何把各章节组合成完整的 AlphaFold 3 模型：加载自带的 7r6r
蛋白、在 CPU / MPS 上端到端推理，最后打印 pLDDT / pTM 分数。


## 0. Setup · 环境

Run from the project root. Adjust `PROJECT` if you cloned elsewhere.
从项目根目录运行；如果路径不同，请修改 `PROJECT`。

In [ ]:
import os, sys, time, json
from pathlib import Path

PROJECT = Path.cwd()
if PROJECT.name != "build-af3-from-scratch":
    # find it walking up
    for p in [PROJECT, *PROJECT.parents]:
        if (p / "solutions").is_dir() and (p / "checkpoints").is_dir():
            PROJECT = p; break

os.environ.setdefault("LAYERNORM_TYPE", "torch")
sys.path.insert(0, str(PROJECT / "solutions"))

CKPT_DIR = PROJECT / "checkpoints"
EXAMPLE  = PROJECT / "solutions" / "examples" / "example.json"
print("project root:", PROJECT)
print("checkpoint dir:", CKPT_DIR, "exists:", CKPT_DIR.is_dir())
print("example JSON :", EXAMPLE, "exists:", EXAMPLE.is_file())


## 1. Build the model · 构建模型

`Protenix` (in `model/model.py`) is the top-level `nn.Module` that wires together
the input embedder, the Pairformer trunk, the diffusion module, and the
confidence head — i.e. Algorithm 1 in the AF3 paper.

`Protenix`（在 `model/model.py`）是顶层 `nn.Module`，把输入嵌入、Pairformer
主干、扩散模块、置信度头拼起来 —— 也就是论文的 Algorithm 1。

In [ ]:
import torch
from copy import deepcopy
from configs.parser import parse_configs
from configs.configs_base import configs as base_cfg
from configs.configs_data import data_configs
from configs.configs_inference import inference_configs
from configs.configs_model_type import model_configs

MODEL_NAME = "protenix_tiny_default_v0.5.0"

def build_config(name):
    cfg = {**base_cfg, **{"data": data_configs}, **inference_configs}
    cfg.update({
        "project": "af3", "run_name": "demo", "base_dir": "/tmp/af3",
        "eval_interval": 0, "log_interval": 0,
        "input_json_path": str(EXAMPLE), "model_name": name,
        "triangle_attention": "torch", "triangle_multiplicative": "torch",
        "enable_tf32": False, "enable_efficient_fusion": False,
    })
    if name in model_configs:
        overrides = deepcopy(model_configs[name])
        def merge(d, s):
            for k, v in s.items():
                if isinstance(v, dict) and isinstance(d.get(k), dict): merge(d[k], v)
                else: d[k] = v
        merge(cfg, overrides)
    return parse_configs(cfg, arg_str=None, fill_required_with_null=True)

cfg = build_config(MODEL_NAME)
cfg.model.N_cycle = 1
cfg.sample_diffusion.N_step = 5
cfg.sample_diffusion.N_sample = 1

from model.model import Protenix
model = Protenix(cfg).eval()
n = sum(p.numel() for p in model.parameters()) / 1e6
print(f"Protenix built — {n:.2f} M parameters")


## 2. Load the official checkpoint · 加载官方权重

`load_state_dict(strict=False)` ignores the one stray `linear_esm.weight` key
that the non-PLM Tiny release leaves behind.

`load_state_dict(strict=False)` 忽略掉非 PLM Tiny 权重里多出来的 `linear_esm.weight`。

In [ ]:
ckpt = torch.load(CKPT_DIR / f"{MODEL_NAME}.pt", map_location="cpu", weights_only=False)
state = ckpt["model"] if "model" in ckpt else ckpt
state = {k.removeprefix("module."): v for k, v in state.items()}
res = model.load_state_dict(state, strict=False)
print(f"missing={len(res.missing_keys)}  unexpected={len(res.unexpected_keys)}")
print("unexpected:", res.unexpected_keys)


## 3. Featurize the input JSON · 特征化输入

`feature_extraction.inference.infer_dataloader.get_inference_dataloader` reads
the bundled `examples/example.json` (a 203-residue protein with pre-computed
MSA), and produces the feature dict the model consumes.

`get_inference_dataloader` 读取自带的 `examples/example.json` (203 残基蛋白 +
预先算好的 MSA)，生成模型可用的特征字典。

In [ ]:
from feature_extraction.inference.infer_dataloader import get_inference_dataloader
from runtime.torch_utils import to_device

loader = get_inference_dataloader(configs=cfg)
batch = next(iter(loader))
data, atom_array, err = batch[0]
assert not err, f"featurization failed: {err}"
print(f"sample: {data['sample_name']}")
print(f"  N_token = {int(data['N_token'])}")
print(f"  N_atom  = {int(data['N_atom'])}")
print(f"  N_msa   = {int(data['N_msa'])}")


## 4. Run inference · 推理

One forward through the model + 5 Euler diffusion steps.
一次前向 + 5 步扩散采样。

In [ ]:
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print("device:", device)

model = model.to(device)
data  = to_device(data, device)

t0 = time.time()
with torch.no_grad():
    pred, _, _ = model(input_feature_dict=data["input_feature_dict"], mode="inference")
print(f"forward time: {time.time() - t0:.2f}s")

summary = pred["summary_confidence"][0]
print(f"  pLDDT         = {float(summary['plddt']):.2f}")
print(f"  pTM           = {float(summary['ptm']):.3f}")
print(f"  ranking score = {float(summary['ranking_score']):.3f}")
print(f"  has_clash     = {bool(summary['has_clash'])}")


## 5. Write the predicted structure · 输出预测结构

In [ ]:
from feature_extraction.utils import save_structure_cif

out_dir = PROJECT / "out_demo"
out_dir.mkdir(exist_ok=True)
cif_path = out_dir / "7r6r_pred.cif"

entity_poly_type = {
    k: v for k, v in data["entity_poly_type"].items() if v != "non-polymer"
}
save_structure_cif(
    atom_array=atom_array,
    pred_coordinate=pred["coordinate"][0],
    output_fpath=str(cif_path),
    entity_poly_type=entity_poly_type,
    pdb_id="7r6r_pred",
)
print("wrote:", cif_path, f"({cif_path.stat().st_size//1024} KB)")


## 6. Visualize · 可视化

If `py3Dmol` is installed, the predicted structure renders inline. Otherwise
open the `.cif` in ChimeraX / PyMOL.

如果安装了 `py3Dmol`，结构会直接在 notebook 里渲染；否则用 ChimeraX / PyMOL 打开
生成的 `.cif`。

In [ ]:
try:
    import py3Dmol
    view = py3Dmol.view(width=600, height=400)
    with open(cif_path) as f:
        view.addModel(f.read(), "mmcif")
    view.setStyle({"cartoon": {"color": "spectrum"}})
    view.zoomTo()
    view.show()
except ImportError:
    print("py3Dmol not installed; skip in-notebook view.")
    print("Open", cif_path, "in ChimeraX / PyMOL.")


## What next? · 接下来

- `solutions/<chapter>/` holds the complete reference implementation.
- `tutorials/<chapter>/` (run `python prepare_tutorials.py` to regenerate) is
  the same files with TODO blocks replaced by `pass` — fill them in to
  reproduce the chapter from scratch.

- `solutions/<chapter>/` 是完整参考实现。
- 运行 `python prepare_tutorials.py` 会生成 `tutorials/<chapter>/`，与
  solutions 同构但 TODO 块被换成 `pass` —— 学生在这里填空。
